# Reading a video with g2ltk

In this notebook, we open one of the reference test videos, print its main features (image size, frame rate, duration) and show some of its frames.

The test videos are:
- `test_gcv.gcv/` : GevCapture video (meta + stamps + raw video)
- `test_mp4.mp4` : lossy compressed video (H.264 codec)
- `test_mkv.mkv` : losslessly compressed video (FFV1 codec)

In [ ]:
%matplotlib widget

import os
import numpy as np
import matplotlib.pyplot as plt

from g2ltk import videoreading as vr
vr.set_verbose('info')

## Selecting the video

There are two ways of selecting a video to use in g2ltk
* Simply provide an `acquisition_path` (path to the video without extension)
* Or define a folder, termed `dataset`, containing the video, termed `acquisition`


In [ ]:
### Simple method: define acquisition_path

acquisition_path = os.path.join('..', 'test_data', 'test_gcv') # 'gcv', 'mp4' or 'mkv'

if not vr.is_this_a_video(acquisition_path):
    raise FileNotFoundError(f'No video found at {acquisition_path}')

In [ ]:
### 'g2ltk' method: define a dataset containing acquisitions

# The *root path* indicates where we are going to find the datasets (directories containing videos)
vr.set_default_root_path('..')

# A *dataset* is the name of a directory containing videos (*acquisitions*) of interest
dataset = 'test_data'
vr.describe_dataset(dataset=dataset, makeitshort=False)

# An *acquisition*  is the name of video file (without extension)
acquisition = 'test_gcv' # 'gcv', 'mp4' or 'mkv'

if not vr.is_this_a_video(acquisition=acquisition, dataset=dataset):
    raise FileNotFoundError(f'No video named {acquisition} in dataset {dataset}')

## Video features

Use `describe_acquisition` to get a list of the features of the chosen video

In [ ]:
vr.describe_acquisition(acquisition_path)

# or

# vr.describe_acquisition(acquisition=acquisition, dataset=dataset)

## Loading the frames

`get_frames` returns an array of shape `(n_frames, height, width)`.
The times of the frames, in seconds, are given by `get_t_s`.

In [ ]:
print(f'Max number of frames: {vr.get_number_of_available_frames(acquisition_path)}')

framenumbers = np.arange(10, 20) # use None to take all frames
framenumbers = vr.format_framenumbers(acquisition_path, framenumbers) # None becomes the array of all the frames

frames = vr.get_frames(acquisition_path, framenumbers=framenumbers)
t_s = vr.get_t_s(acquisition_path, framenumbers=framenumbers)

print(f'Frames array: shape {frames.shape}, dtype {frames.dtype}, values in [{frames.min()}, {frames.max()}]')

## First, middle and last frames

In [ ]:
# frames[i] is the frame number framenumbers[i]: here i is the position of the frame in frames
positions_to_show = [0, len(frames) // 2, len(frames) - 1]

fig, axes = plt.subplots(1, len(positions_to_show), figsize=(9, 3.4), sharex=True, sharey=True, constrained_layout=True)
for ax, i in zip(axes, positions_to_show):
    ax.imshow(frames[i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'frame {framenumbers[i]} (t = {t_s[i]:.3f} s)')
    ax.set_xlabel('x [px]')
axes[0].set_ylabel('z [px]')
plt.show()

## Browsing the video

Use the slider to go through the frames.

In [ ]:
from matplotlib.widgets import Slider

fig, ax = plt.subplots(figsize=(5, 5.5))
fig.subplots_adjust(bottom=0.18)

image = ax.imshow(frames[0], cmap='gray', vmin=0, vmax=255)
ax.set_xlabel('x [px]')
ax.set_ylabel('z [px]')

# the slider goes through the positions in frames
slider_ax = fig.add_axes([0.15, 0.04, 0.7, 0.04])
frame_slider = Slider(slider_ax, 'frame', 0, len(frames) - 1, valinit=0, valstep=1)

def show_frame(position):
    i = int(position)
    image.set_data(frames[i])
    ax.set_title(f'frame {framenumbers[i]} (t = {t_s[i]:.3f} s)')
    frame_slider.valtext.set_text(f'{framenumbers[i]}') # display the frame number next to the slider, not the position
    fig.canvas.draw_idle()

frame_slider.on_changed(show_frame)
show_frame(0)
plt.show()